# NOVA Lab — AMD Instinct MI300X GPU Training Pipeline

This notebook runs on the **AMD AI Academy** JupyterLab environment with MI300X GPUs (192GB HBM3).

## What this does:
1. **Verifies GPU access** (ROCm + PyTorch)
2. **Loads ESM-2 650M** protein language model on GPU
3. **Computes ESM-2 embeddings** for all training proteins (GPU-accelerated)
4. **Trains the ensemble model** (GradientBoosting + XGBoost + RandomForest)
5. **Exports model weights** for download to local backend

### Upload these files to this notebook environment first:
- `4_fireprotDB_bestpH.csv`
- `plddt_cache.json`

## Step 0: Verify AMD MI300X GPU

In [ ]:
!rocm-smi
print("=" * 60)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"ROCm available (torch.cuda): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    raise RuntimeError("No GPU detected! Check ROCm installation.")

## Step 1: Install dependencies

In [ ]:
!pip install fair-esm pandas scikit-learn xgboost scipy --quiet

## Step 2: Load ESM-2 650M on MI300X GPU

The MI300X has 192GB HBM3 — ESM-2 650M (~2.5GB) fits easily. We could even run ESM-2 3B or 15B here.

In [ ]:
import esm
import torch
import numpy as np
import time

print("Loading ESM-2 650M model...")
t0 = time.time()
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model = model.eval().cuda()
print(f"Model loaded on GPU in {time.time()-t0:.1f}s")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Step 3: Load training data

In [ ]:
import pandas as pd
import json

df = pd.read_csv('4_fireprotDB_bestpH.csv')
print(f"FireProtDB: {len(df)} mutations")

# Get unique proteins with sequences
seq_map = {}
for _, r in df.iterrows():
    uid = str(r.get('uniprot_id', '')).strip()
    seq = str(r.get('sequence', '')).strip() if 'sequence' in r else ''
    if uid and uid != 'nan' and seq and len(seq) > 10:
        seq_map[uid] = seq

print(f"Unique proteins with sequences: {len(seq_map)}")

# Load pLDDT cache
try:
    with open('plddt_cache.json') as f:
        plddt_cache = json.load(f)
    print(f"pLDDT cache: {len(plddt_cache)} proteins")
except FileNotFoundError:
    plddt_cache = {}
    print("No pLDDT cache found, will skip pLDDT features")

## Step 4: Compute ESM-2 embeddings on MI300X (GPU-accelerated)

This is the key step that benefits from the MI300X. On CPU this takes hours — on MI300X it takes **minutes**.

In [ ]:
import pickle

esm_cache = {}
proteins = {k: v for k, v in seq_map.items() if v and len(v) > 10}
print(f"Computing ESM-2 embeddings for {len(proteins)} proteins on MI300X...")

t0 = time.time()
for i, (uid, seq) in enumerate(proteins.items()):
    seq_trunc = seq[:1022]
    try:
        data = [("protein", seq_trunc)]
        _, _, tokens = batch_converter(data)
        tokens = tokens.cuda()

        with torch.no_grad():
            results = model(tokens, repr_layers=[33], return_contacts=False)

        reps = results["representations"][33][0, 1:len(seq_trunc)+1].cpu().numpy()
        embeddings = {}
        for pos_idx in range(len(seq_trunc)):
            embeddings[pos_idx + 1] = reps[pos_idx]
        esm_cache[uid] = embeddings
    except Exception as e:
        print(f"  Failed {uid} (len={len(seq)}): {e}")
        esm_cache[uid] = {}

    if (i+1) % 10 == 0 or (i+1) == len(proteins):
        elapsed = time.time() - t0
        rate = (i+1) / elapsed
        eta = (len(proteins) - i - 1) / rate
        print(f"  {i+1}/{len(proteins)} done | {rate:.1f} proteins/sec | ETA {eta:.0f}s")

print(f"\nDone! {len(esm_cache)} proteins embedded in {time.time()-t0:.1f}s")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

with open('esm2_embeddings_mi300x.pkl', 'wb') as f:
    pickle.dump(esm_cache, f)
print("Saved: esm2_embeddings_mi300x.pkl")

## Step 5: Extract features & train ensemble model

In [ ]:
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available, using sklearn only")

VALID_AAS = set('ACDEFGHIKLMNPQRSTVWY')

HYDROPHOBICITY = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8,
    'M': 1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2, 'W': -0.9, 'Y': -1.3,
}
VOLUME = {
    'A': 88.6, 'C': 108.5, 'D': 111.1, 'E': 138.4, 'F': 189.9,
    'G': 60.1, 'H': 153.2, 'I': 166.7, 'K': 168.6, 'L': 166.7,
    'M': 162.9, 'N': 114.1, 'P': 112.7, 'Q': 143.8, 'R': 173.4,
    'S': 89.0, 'T': 116.1, 'V': 140.0, 'W': 227.8, 'Y': 193.6,
}
CHARGE = {'D': -1, 'E': -1, 'K': 1, 'R': 1, 'H': 0.5}
FLEXIBILITY = {
    'G': 1.0, 'S': 0.7, 'D': 0.6, 'N': 0.6, 'P': 0.1,
    'A': 0.4, 'T': 0.5, 'E': 0.5, 'Q': 0.5, 'K': 0.5,
    'R': 0.4, 'H': 0.4, 'M': 0.3, 'L': 0.3, 'V': 0.3,
    'I': 0.2, 'F': 0.2, 'Y': 0.2, 'W': 0.1, 'C': 0.2,
}
HELIX_PROPENSITY = {
    'A': 1.42, 'L': 1.21, 'E': 1.51, 'M': 1.45, 'Q': 1.11,
    'K': 1.16, 'R': 0.98, 'H': 1.00, 'V': 1.06, 'I': 1.08,
    'Y': 0.69, 'C': 0.70, 'W': 1.08, 'F': 1.13, 'T': 0.83,
    'G': 0.57, 'N': 0.67, 'P': 0.57, 'S': 0.77, 'D': 1.01,
}
SHEET_PROPENSITY = {
    'V': 1.70, 'I': 1.60, 'Y': 1.47, 'F': 1.38, 'W': 1.37,
    'L': 1.30, 'T': 1.19, 'C': 1.19, 'Q': 1.10, 'M': 1.05,
    'A': 0.83, 'R': 0.93, 'G': 0.75, 'D': 0.54, 'K': 0.74,
    'S': 0.75, 'H': 0.87, 'N': 0.89, 'P': 0.55, 'E': 0.37,
}
BLOSUM62_DIAG = {
    'A':4,'R':5,'N':6,'D':6,'C':9,'Q':5,'E':5,'G':6,'H':8,'I':4,
    'L':4,'K':5,'M':5,'F':6,'P':7,'S':4,'T':5,'W':11,'Y':7,'V':4
}


def get_esm_features(uid, position):
    embeddings = esm_cache.get(uid, {})
    if not embeddings or position not in embeddings:
        return [0.0] * 20
    emb = embeddings[position]
    f = [
        float(np.mean(emb)), float(np.std(emb)),
        float(np.max(emb)), float(np.min(emb)),
        float(np.median(emb)),
        float(np.percentile(emb, 25)), float(np.percentile(emb, 75)),
        float(np.linalg.norm(emb)),
        float(np.sum(emb > 1.0)), float(np.sum(emb < -1.0)),
        float(stats.skew(emb)), float(stats.kurtosis(emb)),
    ]
    emb_abs = np.abs(emb) + 1e-10
    emb_norm = emb_abs / emb_abs.sum()
    f.append(float(-np.sum(emb_norm * np.log(emb_norm))))
    nbs = [embeddings.get(p) for p in [position-2, position-1, position+1, position+2] if p in embeddings]
    if nbs:
        nm = np.mean(nbs, axis=0)
        f.append(float(np.linalg.norm(emb - nm)))
        f.append(float(np.dot(emb, nm) / (np.linalg.norm(emb) * np.linalg.norm(nm) + 1e-10)))
    else:
        f.extend([0.0, 0.0])
    while len(f) < 20:
        f.append(0.0)
    return f[:20]


def extract_features(row):
    wt = str(row.get('wild_type', '')).strip()
    mut = str(row.get('mutation', '')).strip()
    if wt not in VALID_AAS or mut not in VALID_AAS:
        return None

    dH = HYDROPHOBICITY.get(mut, 0) - HYDROPHOBICITY.get(wt, 0)
    dV = VOLUME.get(mut, 0) - VOLUME.get(wt, 0)
    dC = CHARGE.get(mut, 0) - CHARGE.get(wt, 0)
    dF = FLEXIBILITY.get(mut, 0) - FLEXIBILITY.get(wt, 0)
    dHelix = HELIX_PROPENSITY.get(mut, 1) - HELIX_PROPENSITY.get(wt, 1)
    dSheet = SHEET_PROPENSITY.get(mut, 1) - SHEET_PROPENSITY.get(wt, 1)
    f = [dH, dV, dC, dF, dHelix, dSheet]
    f.extend([abs(dH), abs(dV), abs(dC), abs(dF), abs(dHelix), abs(dSheet)])

    rsa = float(row.get('asa', 0.5)) if pd.notna(row.get('asa')) else 0.5
    rsa = min(1.0, max(0.0, rsa / 200.0))
    burial = 1.0 - rsa
    f.append(rsa)

    cons = float(row.get('conservation', 5)) if pd.notna(row.get('conservation')) else 5.0
    bf = float(row.get('b_factor', 20)) if pd.notna(row.get('b_factor')) else 20.0
    f.extend([cons, bf])
    f.extend([abs(dH) * burial, abs(dV) * burial, abs(dC) * burial])

    cons_wt = BLOSUM62_DIAG.get(wt, 4)
    cons_mut = BLOSUM62_DIAG.get(mut, 4)
    f.extend([cons_wt, cons_mut, cons_wt - cons_mut])

    uid = str(row.get('uniprot_id', '')).strip()
    pos = int(row.get('position', 0))
    plddt_scores = plddt_cache.get(uid, {})
    plddt_val = plddt_scores.get(str(pos), 70.0) / 100.0
    f.extend([plddt_val, 1.0 if plddt_val < 0.5 else 0.0, 1.0 if plddt_val > 0.9 else 0.0])

    esm_feats = get_esm_features(uid, pos)
    f.extend(esm_feats)

    return f

print("Feature extraction ready (41 features per mutation)")

In [ ]:
# Build training dataset
X_list, y_list = [], []
skipped = 0

for _, row in df.iterrows():
    ddg = row.get('ddG')
    if pd.isna(ddg):
        skipped += 1
        continue
    feats = extract_features(row)
    if feats is None:
        skipped += 1
        continue
    X_list.append(feats)
    y_list.append(float(ddg))

X = np.array(X_list)
y = np.array(y_list)
labels = (y < 0).astype(int)

print(f"Training set: {len(X)} mutations")
print(f"Stabilizing (ddG < 0): {sum(labels)}")
print(f"Destabilizing: {len(labels) - sum(labels)}")
print(f"Skipped: {skipped}")
print(f"Feature shape: {X.shape}")

## Step 6: Train ensemble

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Training ensemble models...")
t0 = time.time()

gb = GradientBoostingRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=10, random_state=42
)
gb.fit(X_scaled, y)
print(f"  GradientBoosting trained in {time.time()-t0:.1f}s")

t1 = time.time()
rf = RandomForestRegressor(
    n_estimators=500, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf.fit(X_scaled, y)
print(f"  RandomForest trained in {time.time()-t1:.1f}s")

if HAS_XGB:
    t2 = time.time()
    xgb = XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42,
        device='cuda'
    )
    xgb.fit(X_scaled, y)
    print(f"  XGBoost (MI300X GPU) trained in {time.time()-t2:.1f}s")

    models = [('GradientBoosting', gb), ('XGBoost', xgb), ('RandomForest', rf)]
    weights = [0.35, 0.40, 0.25]
else:
    models = [('GradientBoosting', gb), ('RandomForest', rf)]
    weights = [0.55, 0.45]

print(f"\nTotal training time: {time.time()-t0:.1f}s")

## Step 7: Evaluate — cross-validation

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_maes, cv_pearsons, cv_accs = [], [], []

print("Running 5-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    gb_f = GradientBoostingRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                                     subsample=0.8, min_samples_leaf=10, random_state=42)
    gb_f.fit(X_tr, y_tr)
    rf_f = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_f.fit(X_tr, y_tr)

    if HAS_XGB:
        xgb_f = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
                              random_state=42, device='cuda')
        xgb_f.fit(X_tr, y_tr)
        fold_models = [('GB', gb_f), ('XGB', xgb_f), ('RF', rf_f)]
    else:
        fold_models = [('GB', gb_f), ('RF', rf_f)]

    y_pred_f = np.zeros(len(X_val))
    for (_, m), w in zip(fold_models, weights):
        y_pred_f += m.predict(X_val) * w
    y_pred_f /= sum(weights)

    fold_mae = mean_absolute_error(y_val, y_pred_f)
    fold_r, _ = stats.pearsonr(y_val, y_pred_f)
    fold_acc = np.mean((y_pred_f < 0).astype(int) == (y_val < 0).astype(int))

    cv_maes.append(fold_mae)
    cv_pearsons.append(fold_r)
    cv_accs.append(fold_acc)
    print(f"  Fold {fold+1}: MAE={fold_mae:.4f}, Pearson={fold_r:.4f}, Acc={fold_acc:.1%}")

print(f"\n{'='*50}")
print(f"CV MAE:      {np.mean(cv_maes):.4f} +/- {np.std(cv_maes):.4f}")
print(f"CV Pearson:  {np.mean(cv_pearsons):.4f} +/- {np.std(cv_pearsons):.4f}")
print(f"CV Accuracy: {np.mean(cv_accs):.1%} +/- {np.std(cv_accs):.1%}")

## Step 8: Export model weights

Download these files and copy them into `backend/app/trained_models/` on your local machine.

In [ ]:
import os

OUTPUT_DIR = 'trained_models_mi300x'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save ensemble
ensemble = {'models': models, 'weights': weights}
with open(f'{OUTPUT_DIR}/mutation_regressor.pkl', 'wb') as f:
    pickle.dump(ensemble, f)

# Save scaler
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save ESM embeddings
with open(f'{OUTPUT_DIR}/esm2_embeddings.pkl', 'wb') as f:
    pickle.dump(esm_cache, f)

# Save metadata
meta = {
    'model_type': 'Ensemble (GradientBoosting + XGBoost + RandomForest)',
    'prediction_target': 'DDG (kcal/mol)',
    'n_models': len(models),
    'n_features': X.shape[1],
    'training_samples': len(X),
    'stabilizing_samples': int(sum(labels)),
    'destabilizing_samples': int(len(labels) - sum(labels)),
    'data_sources': {'FireProtDB': len(df)},
    'cv_mae': round(float(np.mean(cv_maes)), 4),
    'cv_pearson': round(float(np.mean(cv_pearsons)), 4),
    'cv_accuracy': round(float(np.mean(cv_accs)), 4),
    'gpu': 'AMD Instinct MI300X',
    'esm_model': 'ESM-2 650M (esm2_t33_650M_UR50D)',
}
with open(f'{OUTPUT_DIR}/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Exported files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{fname}')
    print(f"  {fname}: {size/1e6:.1f} MB")

print(f"\nCopy '{OUTPUT_DIR}/' contents into backend/app/trained_models/ on your local machine.")

## Bonus: Try ESM-2 3B (only possible on MI300X)

MI300X has 192GB VRAM — enough to run ESM-2 3B for much richer protein embeddings. Uncomment to try.

In [ ]:
# # Uncomment to try ESM-2 3B (~12GB VRAM, MI300X has 192GB)
# del model
# torch.cuda.empty_cache()
#
# print("Loading ESM-2 3B model...")
# model_3b, alphabet_3b = esm.pretrained.esm2_t36_3B_UR50D()
# batch_converter_3b = alphabet_3b.get_batch_converter()
# model_3b = model_3b.eval().cuda()
# print(f"ESM-2 3B loaded! GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")